# 02 — BigQuery Loading & Stratified Sampling
## Urban Mobility Intelligence Platform | Chicago TNC 2024

### Research Context
This notebook implements the **data preparation pipeline** described in Section III
of the paper: *"An Explainable Machine Learning Framework for Integrated Surge 
Prediction, Fare Estimation and Demand Forecasting in Post-Pandemic Urban 
Ride-Hailing: A Spatial Equity Analysis"*

This stage corresponds to: **Section III-A: Data Collection & Sampling Strategy**

### Sampling Strategy — Justification
Chicago TNC generates ~200,000+ trips per day across 77 community areas.
The full 2024 dataset contains 91M rows (~20GB) — computationally intractable
for traditional ML training and exceeding cloud free-tier storage limits.

We adopt a **stratified random sampling** strategy:
- **Stratification unit:** Calendar day (366 days in 2024 — leap year)
- **Sample size per day:** 50,000 rows sampled uniformly at random
- **Total dataset:** ~18.3M rows
- **Justification:** Daily stratification ensures:
    1. Full temporal coverage — all seasons, holidays, weekdays/weekends represented
    2. Unbiased fare and demand distributions across the year
    3. No over-representation of high-volume days (e.g. New Year's Eve)
    4. Sufficient daily volume for zone-level demand forecasting

This approach follows established stratified sampling methodology in
transportation data science (Chen et al., 2021; Saadi et al., 2017).

### What this notebook does
1. Loads each monthly CSV (~1.1GB) into a temporary BigQuery table
2. Applies stratified random sampling — 50K rows per day via BigQuery SQL
3. Appends sampled rows to the final `trips_final` table
4. Deletes the temporary table after each month to stay within free tier
5. Verifies final table — row counts, date coverage, zone distribution

### Output
- BigQuery table: `urban-mobility-intel.chicago_tnc.trips_final`
- **~18.3M rows** (366 days × 50,000 rows)
- Perfectly stratified — every day of 2024 equally represented
- Random within each day — all hours, zones, fare ranges captured

In [ ]:

# IMPORTS & CONFIGURATION


import os
import pandas as pd
from google.cloud import bigquery

# --- Credentials ---
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"E:\Projects\ML\Transport-Taxi-Chicago\gcp_credentials.json"

# --- BigQuery config ---
PROJECT_ID      = "urban-mobility-intel"
DATASET_ID      = "chicago_tnc"
TEMP_TABLE      = f"{PROJECT_ID}.{DATASET_ID}.raw_temp"
FINAL_TABLE     = f"{PROJECT_ID}.{DATASET_ID}.trips_final"
ROWS_PER_DAY    = 50000

# --- Local paths ---
RAW_CSV_DIR     = r"E:\Projects\ML\Transport-Taxi-Chicago\data\raw_csv"

# --- Monthly CSV files in order ---
MONTHS = [
    "2024_01", "2024_02", "2024_03", "2024_04",
    "2024_05", "2024_06", "2024_07", "2024_08",
    "2024_09", "2024_10", "2024_11", "2024_12"
]

# --- Connect to BigQuery ---
client = bigquery.Client(project=PROJECT_ID)
print(f"Connected: {client.project}")
print(f"Temp table  : {TEMP_TABLE}")
print(f"Final table : {FINAL_TABLE}")
print(f"Rows per day: {ROWS_PER_DAY:,}")

Connected: urban-mobility-intel
Temp table  : urban-mobility-intel.chicago_tnc.raw_temp
Final table : urban-mobility-intel.chicago_tnc.trips_final
Rows per day: 50,000


## 1. Helper Functions
### 1.1 Load CSV to Temporary BigQuery Table
### 1.2 Stratified Random Sampling Query
### 1.3 Delete Temporary Table

In [3]:
# HELPER FUNCTIONS


def load_csv_to_bigquery_temp(csv_path, temp_table):
    """
    Load a monthly CSV file into a temporary BigQuery table.
    
    - Uses WRITE_TRUNCATE: overwrites any existing temp table
    - Partitions by trip_start_timestamp (DAY) for fast date filtering
    - Clusters by pickup_community_area for fast zone-level queries
    - Fixes dtypes before loading (timestamps, integer zone IDs)
    
    Args:
        csv_path   : Full path to monthly CSV file
        temp_table : BigQuery table ID for temporary storage
    """
    print(f"  Reading {os.path.basename(csv_path)}...")
    df = pd.read_csv(csv_path)
    print(f"  Rows read: {len(df):,}")

    # Fix dtypes
    df['trip_start_timestamp']  = pd.to_datetime(df['trip_start_timestamp'])
    df['trip_end_timestamp']    = pd.to_datetime(df['trip_end_timestamp'])
    df['pickup_community_area'] = df['pickup_community_area'].fillna(0).astype(int)
    df['dropoff_community_area']= df['dropoff_community_area'].fillna(0).astype(int)

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        time_partitioning=bigquery.TimePartitioning(
            type_=bigquery.TimePartitioningType.DAY,
            field="trip_start_timestamp"
        ),
        clustering_fields=["pickup_community_area"],
        autodetect=True
    )

    print(f"  Loading to BigQuery temp table...")
    load_job = client.load_table_from_dataframe(df, temp_table, job_config=job_config)
    load_job.result()

    table = client.get_table(temp_table)
    print(f"  ✓ Temp table loaded: {table.num_rows:,} rows")

    # Free RAM immediately
    del df
    return table.num_rows


def sample_days_to_final(temp_table, final_table, rows_per_day, month_label, is_first_month):
    """
    Sample 50K random rows per day from temp table and append to final table.
    
    Uses QUALIFY + ROW_NUMBER() + RAND() pattern:
    - PARTITION BY DATE(trip_start_timestamp) → groups by calendar day
    - ORDER BY RAND()                         → random shuffle within each day  
    - ROW_NUMBER() <= rows_per_day            → take first 50K from shuffle
    
    This implements simple random sampling within daily strata —
    the core sampling methodology of the paper (Section III-A).
    
    Args:
        temp_table     : Source BigQuery temp table
        final_table    : Destination BigQuery final table
        rows_per_day   : Number of random rows to sample per day (50,000)
        month_label    : e.g. '2024_01' for logging
        is_first_month : If True, WRITE_TRUNCATE; else WRITE_APPEND
    """
    
    write_mode = "WRITE_TRUNCATE" if is_first_month else "WRITE_APPEND"
    print(f"  Sampling {rows_per_day:,} random rows/day → {final_table} ({write_mode})...")

    query = f"""
    CREATE OR REPLACE TABLE `{final_table}`
    PARTITION BY DATE(trip_start_timestamp)
    CLUSTER BY pickup_community_area
    AS
    SELECT * FROM (
        SELECT *
        FROM `{temp_table}`
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY DATE(trip_start_timestamp)
            ORDER BY RAND()
        ) <= {rows_per_day}
    )
    """ if is_first_month else f"""
    INSERT INTO `{final_table}`
    SELECT * FROM (
        SELECT *
        FROM `{temp_table}`
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY DATE(trip_start_timestamp)
            ORDER BY RAND()
        ) <= {rows_per_day}
    )
    """

    job = client.query(query)
    job.result()
    print(f"  ✓ {month_label} sampled and appended to final table")


def delete_temp_table(temp_table):
    """
    Delete temporary BigQuery table after sampling is complete.
    Keeps storage usage within free tier limits.
    """
    client.delete_table(temp_table, not_found_ok=True)
    print(f"  ✓ Temp table deleted")

## 2. Load, Sample & Build Final Table
Processes each month sequentially:
- Loads raw CSV to temporary BigQuery table
- Samples 50K random rows per day
- Appends to `trips_final`
- Deletes temp table before next month

In [8]:
import time

results = {}

for i, month in enumerate(MONTHS):
    csv_path = os.path.join(RAW_CSV_DIR, f"tnc_{month}.csv")

    # skip if file not found
    if not os.path.exists(csv_path):
        print(f"{month} — file not found, skipping")
        results[month] = "missing"
        continue

    # skip if already loaded in trips_final
    if month in months_to_skip:
        print(f"{month} — already in trips_final, skipping")
        results[month] = "skipped"
        continue

    print(f"Processing {month} ({i+1}/12)")

    try:
        # load csv to temp table
        load_csv_to_bigquery_temp(csv_path, TEMP_TABLE)

        # sample 50k per day and append to final table
        sample_days_to_final(
            temp_table     = TEMP_TABLE,
            final_table    = FINAL_TABLE,
            rows_per_day   = ROWS_PER_DAY,
            month_label    = month,
            is_first_month = (i == 0)
        )

        # delete temp table to free storage
        delete_temp_table(TEMP_TABLE)

        results[month] = "done"
        print(f"Done: {month}")

    except Exception as e:
        print(f"Error on {month}: {str(e)}")
        delete_temp_table(TEMP_TABLE)
        results[month] = "error"

print("\nPipeline Summary")
for month, status in results.items():
    print(f"  {month} — {status}")

2024_01 — already in trips_final, skipping
2024_02 — already in trips_final, skipping
2024_03 — already in trips_final, skipping
2024_04 — already in trips_final, skipping
2024_05 — already in trips_final, skipping
2024_06 — already in trips_final, skipping
2024_07 — already in trips_final, skipping
2024_08 — already in trips_final, skipping
2024_09 — already in trips_final, skipping
2024_10 — already in trips_final, skipping
2024_11 — already in trips_final, skipping
2024_12 — already in trips_final, skipping

Pipeline Summary
  2024_01 — skipped
  2024_02 — skipped
  2024_03 — skipped
  2024_04 — skipped
  2024_05 — skipped
  2024_06 — skipped
  2024_07 — skipped
  2024_08 — skipped
  2024_09 — skipped
  2024_10 — skipped
  2024_11 — skipped
  2024_12 — skipped


## 3. Verification
Verify the final table — total rows, date coverage, and daily sample distribution.
This confirms the stratified sampling strategy was applied correctly across all 366 days.

In [9]:
# verify final table

# total rows and date range
summary_query = f"""
SELECT
    COUNT(*)                                    AS total_rows,
    COUNT(DISTINCT DATE(trip_start_timestamp))  AS distinct_days,
    MIN(trip_start_timestamp)                   AS earliest_trip,
    MAX(trip_start_timestamp)                   AS latest_trip,
    COUNT(DISTINCT pickup_community_area)       AS distinct_zones
FROM `{FINAL_TABLE}`
"""

summary = client.query(summary_query).to_dataframe()
print("Final Table Summary")
print(summary.T.to_string())

# rows per month
monthly_query = f"""
SELECT
    EXTRACT(MONTH FROM trip_start_timestamp)    AS month,
    COUNT(*)                                    AS row_count,
    COUNT(DISTINCT DATE(trip_start_timestamp))  AS days_covered
FROM `{FINAL_TABLE}`
GROUP BY 1
ORDER BY 1
"""

monthly = client.query(monthly_query).to_dataframe()
print("\nRows per month")
print(monthly.to_string(index=False))

# daily sample distribution — check min/max rows per day
daily_query = f"""
SELECT
    MIN(daily_count)    AS min_rows_per_day,
    MAX(daily_count)    AS max_rows_per_day,
    AVG(daily_count)    AS avg_rows_per_day
FROM (
    SELECT
        DATE(trip_start_timestamp) AS trip_date,
        COUNT(*)                   AS daily_count
    FROM `{FINAL_TABLE}`
    GROUP BY 1
)
"""

daily = client.query(daily_query).to_dataframe()
print("\nDaily Sample Distribution")
print(daily.to_string(index=False))

Final Table Summary
                                  0
total_rows                 18300000
distinct_days                   366
earliest_trip   2024-01-01 00:00:00
latest_trip     2024-12-31 23:45:00
distinct_zones                   78

Rows per month
 month  row_count  days_covered
     1    1550000            31
     2    1450000            29
     3    1550000            31
     4    1500000            30
     5    1550000            31
     6    1500000            30
     7    1550000            31
     8    1550000            31
     9    1500000            30
    10    1550000            31
    11    1500000            30
    12    1550000            31

Daily Sample Distribution
 min_rows_per_day  max_rows_per_day  avg_rows_per_day
            50000             50000           50000.0


## 4. Idempotency Check
Safety check to prevent duplicate data if this notebook is rerun.
Queries which months already exist in `trips_final` and skips them.
Run this cell before re-executing the pipeline in Cell 6.

In [7]:
# check which months are already loaded in trips_final

already_loaded_query = f"""
SELECT
    EXTRACT(MONTH FROM trip_start_timestamp) AS month,
    COUNT(*) AS row_count
FROM `{FINAL_TABLE}`
GROUP BY 1
ORDER BY 1
"""

try:
    already_loaded = client.query(already_loaded_query).to_dataframe()
    loaded_months = already_loaded['month'].tolist()
    
    print("Months already in trips_final:")
    print(already_loaded.to_string(index=False))
    
    # map month numbers to MONTHS list indices
    months_to_skip = [MONTHS[m-1] for m in loaded_months]
    print(f"\nThese months will be skipped if pipeline is rerun:")
    print(months_to_skip)
    
except Exception as e:
    print(f"trips_final does not exist yet or is empty: {e}")
    months_to_skip = []

Months already in trips_final:
 month  row_count
     1    1550000
     2    1450000
     3    1550000
     4    1500000
     5    1550000
     6    1500000
     7    1550000
     8    1550000
     9    1500000
    10    1550000
    11    1500000
    12    1550000

These months will be skipped if pipeline is rerun:
['2024_01', '2024_02', '2024_03', '2024_04', '2024_05', '2024_06', '2024_07', '2024_08', '2024_09', '2024_10', '2024_11', '2024_12']
